# Phase 2: BiLSTM-CRF baseline (Hindi)

Owner: Sajan Kumar. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 2.

**Model** (Lample et al. 2016): each word becomes a fastText Hindi vector plus a character-level BiLSTM feature (which helps with rare and unseen words). A word-level BiLSTM reads the whole sentence. A **CRF** layer then picks the best *sequence* of tags, using learned tag-to-tag transition scores.

**Gate:** the Hindi gold entity F1 is logged in `results/results.csv`. The expected range is mid-to-high 60s. This is a baseline, so don't chase SOTA.

**How to run**
1. Open this notebook in **Colab** (dev/debug) or **Kaggle** (long run) with a **GPU** runtime.
2. Run it once with `DEBUG = True`: 5k sentences and 2 epochs, finishing in a few minutes, to confirm the pipeline works end to end.
3. Set `DEBUG = False` and *Run all*. Checkpoints are saved every epoch to Drive or Kaggle. If the session dies, just *Run all* again and it resumes.
4. Copy the logged row into `results/results.csv` in the repo and commit it (see the last cell).

Shared code comes from `src/` (loader, label map, seqeval evaluator, results logger). Do **not** redefine any of it here.

In [1]:
# Pin versions (docs/04-compute.md). Colab/Kaggle already ship torch + numpy.
%pip install -q pytorch-crf==0.7.2 seqeval==1.2.2  # Install the CRF layer package and the seqeval NER-metric library, quietly (-q)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os  # Filesystem path checks and joins
import subprocess  # To shell out to `git clone` when running on Colab/Kaggle
import sys  # To modify sys.path so the repo's `src` package can be imported

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"  # Canonical clone URL for this project

# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook.
if os.path.isdir("../src"):  # Running locally from notebooks/, so src/ is one level up
    REPO_ROOT = os.path.abspath("..")  # Resolve to an absolute path for reliable imports/joins
elif os.path.isdir("src"):  # Running from the repo root itself
    REPO_ROOT = os.path.abspath(".")  # Current directory is already the repo root
else:  # Neither local checkout nor repo root found - must be on Colab/Kaggle without the repo present yet
    REPO_ROOT = os.path.abspath("NaamRekha")  # Expect the repo to be cloned into a "NaamRekha" subfolder
    if not os.path.isdir(REPO_ROOT):  # Not cloned yet (fresh Colab/Kaggle session)
        # If the repo is private, use https://<token>@github.com/... or upload src/ manually.
        url = REPO_URL  # Default to the plain public clone URL
        token = None  # Will hold a GitHub token if one is found/entered
        try:  # Attempt to read a stored token from Colab's Secrets, if running on Colab
            from google.colab import userdata  # Colab-only API for reading stored secrets
            token = userdata.get("GH_TOKEN")      # Colab Secret, if available
        except Exception:  # Not on Colab, or no secret configured - fall through silently
            pass  # Not running on Colab, or no secret configured - fall through silently
        if not token:  # No Colab secret found - ask the user directly instead
            from getpass import getpass  # Prompt for a token without echoing it to notebook output
            token = getpass("GitHub token (Enter to skip if repo is public): ")  # Ask the user interactively
        if token:  # A token was supplied (Colab secret or manual entry) - use it for auth
            url = REPO_URL.replace("https://", f"https://{token}@")  # Embed the token for authenticated HTTPS clone
        subprocess.run(["git", "clone", "--depth", "1", url, REPO_ROOT], check=True)  # Shallow-clone the repo; raise if it fails
sys.path.insert(0, REPO_ROOT)  # Make the repo root importable so `from src import ...` below works

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution  # Shared helpers: label set, seqeval evaluator, data loader, results logger, tag stats

print("repo:", REPO_ROOT)  # Confirm which path is being used as the repo root
print("labels:", LABELS)  # Confirm the label set loaded correctly (should match Phase 1's IOB2 tags)

repo: /content/NaamRekha
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Config

Every knob is in this one cell. Record anything you change in the `notes` of the results row (that happens automatically for the main ones).

In [3]:
SEED = 42  # Fixed random seed so sampling, shuffling and weight init are reproducible across runs
LANG = "hi"  # Hindi is the language being modeled in this phase

DEBUG = False             # True: 5k sentences, 2 epochs (pipeline check). False: the real run.
SUBSET_SIZE = 100_000    # training sentences sampled from the ~985k silver train set
EPOCHS = 20              # upper bound; early stopping on val F1 usually ends sooner
PATIENCE = 3             # stop after this many epochs without a val-F1 improvement

BATCH_SIZE = 32  # Number of sentences per training batch
LR = 1e-3                # Adam
CLIP = 5.0               # gradient-norm clipping
DROPOUT = 0.5  # Dropout probability applied after the embeddings and BiLSTM outputs
WORD_DIM = 300           # fastText cc.hi.300
CHAR_DIM = 30  # Dimensionality of learned character embeddings
CHAR_HIDDEN = 25         # per direction
HIDDEN = 128             # word BiLSTM, per direction
MAX_WORD_CHARS = 20      # words longer than this are truncated for the char BiLSTM
WORD_DROPOUT = 0.5       # chance to replace a singleton train word with <unk> (teaches <unk>)

USE_FASTTEXT = True  # Whether to initialize word embeddings from pretrained fastText vectors
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.vec.gz"  # Source of the pretrained Hindi fastText vectors

if DEBUG:  # In debug mode, shrink the run drastically so the whole pipeline can be smoke-tested quickly
    SUBSET_SIZE, EPOCHS = 5_000, 2  # Small subset and few epochs just to confirm nothing crashes end to end

In [4]:
def pick_storage() -> str:  # Decide where to persist checkpoints/embeddings depending on the runtime
    # Persistent storage for checkpoints: Kaggle working dir, Google Drive, or local checkpoints/.
    if os.path.isdir("/kaggle/working"):  # Kaggle provides a persistent working directory
        return "/kaggle/working/naamrekha"  # Use Kaggle's own persistent path
    try:  # Attempt to detect and mount Google Drive (only succeeds on Colab)
        from google.colab import drive  # type: ignore  # Colab-only module for mounting Google Drive
        drive.mount("/content/drive")  # Mount Drive so checkpoints survive Colab session resets
        return "/content/drive/MyDrive/NaamRekha"  # Store checkpoints under the user's Drive
    except ImportError:  # google.colab is not importable - not running on Colab
        return os.path.join(REPO_ROOT, "checkpoints")  # Fallback: plain local run, use the repo's checkpoints/ folder


STORAGE = pick_storage()  # Resolve the storage root once for this run
RUN_NAME = f"bilstm-crf_{LANG}_n{SUBSET_SIZE}_s{SEED}" + ("_debug" if DEBUG else "")  # Unique name encoding model, language, subset size, seed, and debug flag
CKPT_DIR = os.path.join(STORAGE, "phase2", RUN_NAME)  # Directory for this specific run's checkpoints
FT_DIR = os.path.join(STORAGE, "fasttext")  # Shared directory for cached fastText vectors (reused across runs)
os.makedirs(CKPT_DIR, exist_ok=True)  # Create the checkpoint directory if it doesn't already exist
os.makedirs(FT_DIR, exist_ok=True)  # Create the fastText cache directory if it doesn't already exist
print("checkpoints ->", CKPT_DIR)  # Log where this run's artifacts will be saved

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/NaamRekha/phase2/bilstm-crf_hi_n100000_s42


In [5]:
import random  # Python's RNG, used for seeding and the subset sampling

import numpy as np  # NumPy RNG seeding and array operations elsewhere in the notebook
import torch  # PyTorch, the deep learning framework used for the model

random.seed(SEED)  # Seed Python's random module for reproducible sampling
np.random.seed(SEED)  # Seed NumPy's RNG for reproducible array operations
torch.manual_seed(SEED)  # Seed PyTorch's CPU RNG for reproducible weight init / dropout
torch.cuda.manual_seed_all(SEED)  # Seed all CUDA devices too, for reproducibility on GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Prefer GPU if available, else fall back to CPU
print("torch", torch.__version__, "| device:", device)  # Report the PyTorch version and chosen device
if device.type == "cpu":  # Warn if no GPU was found
    print("WARNING: no GPU. Fine for DEBUG, too slow for the real run.")  # Training on CPU is impractical for the full (non-debug) run

torch 2.14.1+cu130 | device: cuda


## Data

Train on a seeded random **subset** of the silver train set. Pick the best epoch on **val** (also silver). Report on the **full gold test** set, and never touch test for any decision.

In [6]:
train_full = load_split(LANG, "train")  # Load the full silver training split (~985k sentences) via the shared src loader
val = load_split(LANG, "val")  # Load the silver validation split, used for model selection/early stopping
test = load_split(LANG, "test")  # Load the gold test split, used only for final reporting

n = min(SUBSET_SIZE, len(train_full))  # Don't request more sentences than actually exist in train_full
idx = sorted(random.Random(SEED).sample(range(len(train_full)), n))  # Seeded random sample of sentence indices, sorted for deterministic ordering
train = [train_full[i] for i in idx]  # Materialize the sampled subset as the actual training set
del train_full  # Free the large full training list now that only the subset is needed

print(f"train {len(train):,} (subset) | val {len(val):,} | test {len(test):,}")  # Report split sizes to sanity-check the subsampling


def entity_report(rows, name):  # rows: list of (tokens, tags); name: label used when printing
    # Entity density check (docs/03-conventions.md): make sure the subset isn't mostly O.
    counts = tag_distribution(rows)  # Frequency of each tag (e.g. O, B-PER, I-PER, ...) across the split
    total = sum(counts.values())  # Total number of tag/token occurrences
    ents = {t: counts.get(f"B-{t}", 0) for t in ("PER", "ORG", "LOC")}  # Count of entity-start tags per entity type (one B- per entity mention)
    with_ent = sum(any(t != "O" for t in tags) for _, tags in rows)  # Number of sentences containing at least one non-O tag
    print(f"{name}: entities {ents} | O = {counts['O'] / total:.1%} of tokens | "  # First half of the summary line (implicitly concatenated with the next)
          f"{with_ent / len(rows):.1%} of sentences have >=1 entity")  # Summarize entity density for this split


entity_report(train, "train subset")  # Check entity density in the sampled training subset
entity_report(test, "gold test")  # Check entity density in the gold test set for comparison

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


data/hi_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

data/hi_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

train 100,000 (subset) | val 13,460 | test 864
train subset: entities {'PER': 78044, 'ORG': 70041, 'LOC': 74368} | O = 82.0% of tokens | 90.6% of sentences have >=1 entity
gold test: entities {'PER': 788, 'ORG': 521, 'LOC': 613} | O = 83.0% of tokens | 86.0% of sentences have >=1 entity


## Vocabulary and fastText embeddings

- Word vocab = every word in the train subset, plus val/test words that have a fastText vector. Pretrained vectors are unsupervised, so using them for test words is standard practice (Lample et al. do the same) and leaks no labels.
- Words with no fastText vector start random and are learned.
- The full `.vec.gz` is about 1.2 GB. It is downloaded once to a local cache, and only the vectors we need are saved to persistent storage.

In [7]:
import gzip  # To stream-read the gzip-compressed fastText vectors file without fully decompressing to disk
import urllib.request  # To download the fastText vectors file over HTTP
from collections import Counter  # For counting word/char frequencies

from tqdm.auto import tqdm  # Progress bar for the (slow) scan over the fastText vectors file

PAD, UNK = "<pad>", "<unk>"  # Reserved vocabulary tokens: padding (id 0) and unknown/out-of-vocabulary (id 1)

word_counts = Counter(w for words, _ in train for w in words)  # Frequency of every word in the sampled training subset
wanted = set(word_counts) | {w for rows in (val, test) for words, _ in rows for w in words}  # All words we might need a fastText vector for: train + val + test vocab


def load_fasttext_subset(wanted_words: set) -> dict:  # Returns {word: vector} only for words actually needed, avoiding loading the full 1.2GB file into memory
    cache = os.path.join(FT_DIR, f"cc.{LANG}.300.filtered_{RUN_NAME}.pt")  # Path to a cached, already-filtered copy of the needed vectors for this run
    if os.path.exists(cache):  # Reuse the filtered cache if a previous run already built it
        print("fastText: using cached subset", cache)  # Avoid re-scanning the huge file when not necessary
        return torch.load(cache)  # Load the cached {word: tensor} dict directly
    raw = os.path.join(os.path.expanduser("~/.cache/naamrekha"), os.path.basename(FASTTEXT_URL))  # Local path for the full downloaded fastText file
    os.makedirs(os.path.dirname(raw), exist_ok=True)  # Ensure the cache directory exists before downloading
    if not os.path.exists(raw):  # Only download once; reuse across runs/sessions if already present
        print("fastText: downloading", FASTTEXT_URL, "(~1.2 GB, one time)")  # Warn this is a large, slow download
        urllib.request.urlretrieve(FASTTEXT_URL, raw)  # Fetch the full gzip-compressed vectors file
    vecs = {}  # Will hold {word: tensor} only for words present in wanted_words
    with gzip.open(raw, "rt", encoding="utf-8") as f:  # Stream-decompress the file as text, line by line
        total, _dim = map(int, f.readline().split())  # First line of fastText .vec files is "<num_words> <dim>"; dim unused since WORD_DIM is fixed
        for line in tqdm(f, total=total, desc="scan fastText"):  # Scan every vector line, showing progress against the known total
            word, rest = line.rstrip("\n").split(" ", 1)  # Split each line into the word and its space-separated vector values
            if word in wanted_words:  # Only keep vectors for words that appear in our train/val/test vocab
                vecs[word] = torch.tensor(np.asarray(rest.split(), dtype=np.float32))  # Parse the remaining floats into a tensor
    torch.save(vecs, cache)  # Cache the filtered subset so future runs skip the full scan
    return vecs  # Return the filtered {word: vector} mapping


ft = load_fasttext_subset(wanted) if USE_FASTTEXT else {}  # Load pretrained vectors only if USE_FASTTEXT is enabled; else start with none

# Deterministic order: by frequency, then alphabetically.
train_words = sorted(word_counts, key=lambda w: (-word_counts[w], w))  # Vocabulary built only from train words, most frequent first (ties broken alphabetically)
extra_words = sorted(w for w in wanted - set(word_counts) if w in ft)  # Val/test-only words added, but only if we have a pretrained vector for them (so nothing new is "learned" from them)
itos = [PAD, UNK] + train_words + extra_words  # Full index-to-string vocabulary list; PAD=0, UNK=1 by construction
stoi = {w: i for i, w in enumerate(itos)}  # String-to-index lookup, inverse of itos

emb = torch.empty(len(itos), WORD_DIM).normal_(0, 0.1, generator=torch.Generator().manual_seed(SEED))  # Randomly initialize the embedding matrix (std=0.1), seeded for reproducibility
emb[0] = 0.0  # Zero out the PAD embedding so padding never contributes signal
hits = 0  # Counts how many vocabulary words were actually found in fastText
for w, i in stoi.items():  # Overwrite random init with pretrained vectors wherever available
    if w in ft:  # Only overwrite if this vocabulary word actually has a pretrained fastText vector
        emb[i] = ft[w]  # Replace this word's random vector with its fastText vector
        hits += 1  # Track coverage of the pretrained embeddings

singleton_ids = torch.tensor([stoi[w] for w, c in word_counts.items() if c == 1])  # IDs of words appearing exactly once in training - used later for word dropout to <unk>

char_counts = Counter(c for w in word_counts for c in w)  # Frequency of every individual character across the training vocabulary
chars = [PAD, UNK] + sorted(char_counts, key=lambda c: (-char_counts[c], c))  # Character vocabulary, most frequent first, with PAD/UNK reserved at the start
ctoi = {c: i for i, c in enumerate(chars)}  # Character-to-index lookup for the char-level BiLSTM input


def oov_rate(rows):  # Fraction of tokens in `rows` that fall outside the word vocabulary (map to <unk>)
    toks = [w for words, _ in rows for w in words]  # Flatten all tokens across all sentences in the split
    return sum(w not in stoi for w in toks) / len(toks)  # Proportion of tokens not found in stoi (i.e. treated as unknown)


print(f"vocab {len(itos):,} words ({hits:,} with fastText = {hits / len(itos):.1%}) | {len(chars)} chars")  # Report vocabulary size and fastText coverage
print(f"test token OOV rate: {oov_rate(test):.2%}")  # Report how many gold test tokens are out-of-vocabulary (a key difficulty indicator)

fastText: using cached subset /content/drive/MyDrive/NaamRekha/fasttext/cc.hi.300.filtered_bilstm-crf_hi_n100000_s42.pt
vocab 98,952 words (77,233 with fastText = 78.1%) | 555 chars
test token OOV rate: 0.73%


## Batching

In [8]:
from torch.utils.data import DataLoader  # PyTorch's batching/shuffling utility


def encode(words, tags):  # Convert one sentence's raw tokens/tags into integer-id sequences the model can consume
    w = [stoi.get(x, 1) for x in words]  # Map each word to its vocab id, defaulting to 1 (<unk>) if not found
    c = [[ctoi.get(ch, 1) for ch in x[:MAX_WORD_CHARS]] for x in words]  # Per-word list of character ids, truncated to MAX_WORD_CHARS
    t = [label2id[x] for x in tags]  # Map each IOB2 tag string to its integer label id
    return w, c, t  # Return parallel (word ids, char id lists, tag ids) for this sentence


def collate(batch):  # Pad a list of variable-length encoded sentences into fixed-size tensors for a batch
    B = len(batch)  # Batch size: number of sentences in this batch
    T = max(len(w) for w, _, _ in batch)  # Longest sentence (in words) in this batch - used as the padded sequence length
    L = max(len(cw) for _, c, _ in batch for cw in c)  # Longest word (in characters) in this batch - used as the padded char length
    words = torch.zeros(B, T, dtype=torch.long)  # Word-id tensor, zero-padded (0 = PAD)
    tags = torch.zeros(B, T, dtype=torch.long)  # Tag-id tensor, zero-padded (padding positions are masked out below anyway)
    mask = torch.zeros(B, T, dtype=torch.bool)  # Boolean mask marking which positions are real tokens vs padding
    chars_t = torch.zeros(B, T, L, dtype=torch.long)  # Character-id tensor per word, zero-padded
    char_lens = torch.zeros(B, T, dtype=torch.long)  # Actual character length of each word, needed to pack the char BiLSTM input
    for b, (w, c, t) in enumerate(batch):  # Fill in the padded tensors sentence by sentence
        words[b, : len(w)] = torch.tensor(w)  # Place this sentence's word ids into the padded row
        tags[b, : len(t)] = torch.tensor(t)  # Place this sentence's tag ids into the padded row
        mask[b, : len(w)] = True  # Mark the real (non-padding) positions for this sentence
        for j, cw in enumerate(c):  # Fill character ids word by word within the sentence
            chars_t[b, j, : len(cw)] = torch.tensor(cw)  # Place this word's character ids into the padded slot
            char_lens[b, j] = len(cw)  # Record the true character length for packing
    return words, chars_t, char_lens, tags, mask  # Batched tensors ready for the model


train_enc = [encode(w, t) for w, t in train]  # Pre-encode every training sentence once, up front
val_enc = [encode(w, t) for w, t in val]  # Pre-encode every validation sentence
test_enc = [encode(w, t) for w, t in test]  # Pre-encode every test sentence

## Model: char-BiLSTM + word-BiLSTM + CRF

The CRF is left **unconstrained**, so it does not forbid `O -> I-X`. The silver data contains about 0.5% orphan `I-` tags, which we keep as-is per the Phase 1 contract, so the CRF should learn the transitions from the data.

In [9]:
import torch.nn as nn  # PyTorch neural network building blocks (Embedding, LSTM, Linear, Dropout, ...)
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence  # Utilities to efficiently run RNNs over variable-length, padded sequences
from torchcrf import CRF  # Conditional Random Field layer providing structured sequence decoding/loss


class BiLSTMCRF(nn.Module):  # Full model: char-BiLSTM features + word-BiLSTM + CRF output layer
    def __init__(self, word_emb: torch.Tensor, n_chars: int, n_tags: int):  # word_emb: pretrained/initialized embedding matrix; n_chars/n_tags: vocab sizes
        super().__init__()  # Standard nn.Module initialization
        self.word_emb = nn.Embedding.from_pretrained(word_emb, freeze=False, padding_idx=0)  # Word embedding layer, initialized from `emb` and fine-tuned during training (freeze=False)
        self.char_emb = nn.Embedding(n_chars, CHAR_DIM, padding_idx=0)  # Learned character embedding table
        self.char_lstm = nn.LSTM(CHAR_DIM, CHAR_HIDDEN, batch_first=True, bidirectional=True)  # Character-level BiLSTM producing a fixed-size feature per word
        self.lstm = nn.LSTM(word_emb.size(1) + 2 * CHAR_HIDDEN, HIDDEN, batch_first=True, bidirectional=True)  # Word-level BiLSTM over [word_emb ; char_feature] concatenation
        self.dropout = nn.Dropout(DROPOUT)  # Shared dropout layer applied at a couple of points for regularization
        self.proj = nn.Linear(2 * HIDDEN, n_tags)  # Projects BiLSTM hidden states (both directions concatenated) to per-tag emission scores
        self.crf = CRF(n_tags, batch_first=True)  # CRF layer: learns tag-to-tag transition scores and does Viterbi decoding

    def char_features(self, chars, char_lens):  # Compute one char-BiLSTM-derived feature vector per word in the batch
        # One vector per word: final forward + backward states of the char BiLSTM.
        B, T, L = chars.shape  # Batch size, sentence length (words), max word length (chars)
        flat, lens = chars.view(B * T, L), char_lens.view(-1)  # Flatten (batch, word) into one dimension so every word is processed independently
        out = chars.new_zeros(B * T, 2 * CHAR_HIDDEN, dtype=torch.float)  # Output buffer for all words' char features, zero for padding words
        real = lens > 0  # skip padding positions
        packed = pack_padded_sequence(self.char_emb(flat[real]), lens[real].cpu(),  # Embed the real (non-padding) characters, then pack them for the LSTM
                                      batch_first=True, enforce_sorted=False)  # Pack only the real (non-padding) words' character embeddings for efficient LSTM processing
        _, (h, _) = self.char_lstm(packed)  # Run the char BiLSTM; h holds the final hidden state for each direction
        out[real] = torch.cat([h[0], h[1]], dim=-1)  # Concatenate forward (h[0]) and backward (h[1]) final states as the word's char feature
        return out.view(B, T, -1)  # Reshape back to (batch, sentence_length, char_feature_dim)

    def emissions(self, words, chars, char_lens, mask):  # Compute per-tag emission scores for every token in the batch
        x = torch.cat([self.word_emb(words), self.char_features(chars, char_lens)], dim=-1)  # Concatenate word embeddings with char-level features
        x = self.dropout(x)  # Regularize the combined word representation
        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)  # Pack by true sentence lengths (derived from the mask) for the word BiLSTM
        h, _ = self.lstm(packed)  # Run the word-level BiLSTM over the sentence
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=words.size(1))  # Unpack back to a fixed-length padded tensor matching the input width
        return self.proj(self.dropout(h))  # Dropout then project BiLSTM outputs to per-tag emission scores

    def loss(self, words, chars, char_lens, tags, mask):  # Training objective: negative log-likelihood of the gold tag sequence under the CRF
        return -self.crf(self.emissions(words, chars, char_lens, mask), tags, mask=mask, reduction="mean")  # CRF forward returns log-likelihood; negate and average for a loss to minimize

    def decode(self, words, chars, char_lens, mask):  # Inference: find the best-scoring tag sequence per sentence (Viterbi decoding)
        return self.crf.decode(self.emissions(words, chars, char_lens, mask), mask=mask)  # Returns a list of tag-id lists, one per sentence, respecting the mask


model = BiLSTMCRF(emb, len(chars), len(LABELS)).to(device)  # Instantiate the model with the built embeddings/vocab sizes, move to GPU/CPU
optimizer = torch.optim.Adam(model.parameters(), lr=LR)  # Adam optimizer over all model parameters
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")  # Report total parameter count as a sanity check on model size

30,207,032 parameters


## Train (checkpoint every epoch, resume automatically)

- `last.pt` is the state after the latest epoch (model, optimizer, history). If it exists, training resumes from it.
- `best.pt` holds the weights with the best **val** F1, plus the vocab, so it can be reused later.

In [10]:
import time  # To measure per-epoch wall-clock duration


@torch.no_grad()  # Disable gradient tracking during inference for speed/memory
def predict(enc_rows, batch_size=256):  # Run the model over pre-encoded rows and return predicted tag sequences as strings
    model.eval()  # Switch to eval mode (disables dropout)
    preds = []  # Will accumulate predicted tag-string sequences across all batches
    for words, chars_t, char_lens, _, mask in DataLoader(enc_rows, batch_size=batch_size, collate_fn=collate):  # Iterate batches without shuffling (order doesn't matter for prediction)
        out = model.decode(words.to(device), chars_t.to(device), char_lens.to(device), mask.to(device))  # Viterbi-decode the best tag-id sequence per sentence in the batch
        preds.extend([id2label[i] for i in seq] for seq in out)  # Convert predicted tag ids back to their string labels
    return preds  # List of predicted tag sequences, aligned with the input rows' order


def score(rows, enc_rows):  # Compute seqeval metrics comparing gold tags (from `rows`) to model predictions (from `enc_rows`)
    return evaluate([tags for _, tags in rows], predict(enc_rows))  # `evaluate` wraps seqeval to compute precision/recall/F1


LAST, BEST = os.path.join(CKPT_DIR, "last.pt"), os.path.join(CKPT_DIR, "best.pt")  # Paths for the "resume" checkpoint and the "best-so-far" checkpoint
start_epoch, best_f1, best_epoch, bad_epochs, history = 0, -1.0, 0, 0, []  # Initial training state: no epochs run yet, no best F1 found yet

if os.path.exists(LAST):  # If a previous run left off partway through, resume from it
    st = torch.load(LAST, map_location=device)  # Load the saved training state
    model.load_state_dict(st["model"])  # Restore model weights
    optimizer.load_state_dict(st["optimizer"])  # Restore optimizer state (e.g. Adam's moving averages)
    start_epoch, best_f1, best_epoch = st["epoch"], st["best_f1"], st["best_epoch"]  # Restore epoch counters and best-F1 tracking
    bad_epochs, history = st["bad_epochs"], st["history"]  # Restore early-stopping counter and per-epoch metric history
    print(f"Resumed after epoch {start_epoch} (best val F1 {best_f1:.4f} @ epoch {best_epoch})")  # Confirm the resume point

singletons = singleton_ids.to(device)  # Move the precomputed singleton-word ids to the training device, for word dropout

for epoch in range(start_epoch + 1, EPOCHS + 1):  # Main training loop, continuing from start_epoch if resumed
    if bad_epochs >= PATIENCE:  # Early-stopping check: stop if val F1 hasn't improved for `PATIENCE` epochs
        print("Early stopping already triggered.")  # Inform that no further training will occur this run
        break  # Exit the training loop
    model.train()  # Switch to training mode (enables dropout)
    t0, total = time.time(), 0.0  # Track epoch start time and running loss sum
    g = torch.Generator().manual_seed(SEED + epoch)  # reproducible shuffle, even after resume
    loader = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=g)  # Fresh shuffled batches each epoch, deterministically seeded
    for words, chars_t, char_lens, tags, mask in tqdm(loader, desc=f"epoch {epoch}", leave=False):  # Iterate training batches with a progress bar
        words, chars_t, char_lens = words.to(device), chars_t.to(device), char_lens.to(device)  # Move batch tensors to the training device
        tags, mask = tags.to(device), mask.to(device)  # Move remaining batch tensors to the training device
        if WORD_DROPOUT > 0:  # Optionally apply word dropout to help the model learn to handle <unk> at test time
            drop = torch.isin(words, singletons) & (torch.rand(words.shape, device=device) < WORD_DROPOUT)  # Randomly select singleton-word positions to drop, at the configured rate
            words = words.masked_fill(drop, 1)  # 1 = <unk>
        loss = model.loss(words, chars_t, char_lens, tags, mask)  # Compute the CRF negative log-likelihood loss for this batch
        optimizer.zero_grad()  # Clear gradients from the previous step
        loss.backward()  # Backpropagate to compute gradients
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)  # Clip gradient norm to stabilize training (common for LSTMs)
        optimizer.step()  # Apply the optimizer update to model weights
        total += loss.item()  # Accumulate scalar loss for averaging later

    val_scores = score(val, val_enc)  # Evaluate on the validation set at the end of the epoch
    history.append({"epoch": epoch, "loss": total / len(loader), **val_scores})  # Record this epoch's average loss and validation metrics
    improved = val_scores["F1_overall"] > best_f1  # Check whether this epoch beats the best validation F1 so far
    if improved:  # New best model found
        best_f1, best_epoch, bad_epochs = val_scores["F1_overall"], epoch, 0  # Update best tracking and reset the early-stopping counter
        torch.save({"model": model.state_dict(), "itos": itos, "chars": chars, "labels": LABELS,  # Save weights plus everything needed to reconstruct the vocab/labels later
                    "epoch": epoch, "val_scores": val_scores}, BEST)  # Persist the best checkpoint, including vocab needed to reload the model later
    else:  # This epoch did not beat the best validation F1
        bad_epochs += 1  # No improvement this epoch; count toward early stopping
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch,  # Snapshot everything needed to resume training exactly where it left off
                "best_f1": best_f1, "best_epoch": best_epoch, "bad_epochs": bad_epochs,  # Include early-stopping bookkeeping in the resume state
                "history": history}, LAST)  # Always persist the "resume" checkpoint after every epoch, regardless of improvement
    print(f"epoch {epoch:2d} | loss {total / len(loader):.3f} | val F1 {val_scores['F1_overall']:.4f}"  # First half of the per-epoch summary line
          f"{' *' if improved else ''} | {time.time() - t0:.0f}s")  # Log epoch summary, marking new-best epochs with '*' 

Resumed after epoch 1 (best val F1 0.7424 @ epoch 1)


epoch 2:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  2 | loss 2.973 | val F1 0.7484 * | 372s


epoch 3:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  3 | loss 2.566 | val F1 0.7517 * | 386s


epoch 4:   0%|          | 0/3125 [00:00<?, ?it/s]

... (3 outputs left out)

epoch 6:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  6 | loss 1.937 | val F1 0.7402 | 395s


epoch 7:   0%|          | 0/3125 [00:00<?, ?it/s]

epoch  7 | loss 1.803 | val F1 0.7447 | 397s
Early stopping already triggered.


## Evaluate the best checkpoint on Hindi gold test

In [11]:
from seqeval.metrics import classification_report  # Detailed per-entity-type precision/recall/F1 report

model.load_state_dict(torch.load(BEST, map_location=device)["model"])  # Reload the best-validation-F1 checkpoint's weights before final evaluation
val_scores = score(val, val_enc)  # Re-confirm validation score using the best checkpoint (should match the value recorded during training)
test_preds = predict(test_enc)  # Generate predictions on the gold test set
test_scores = evaluate([tags for _, tags in test], test_preds)  # Compute overall seqeval metrics (precision/recall/F1) on gold test

print(f"best epoch {best_epoch} | val (silver) F1 {val_scores['F1_overall']:.4f}")  # Report which epoch was selected and its validation score
print("TEST (gold):", {k: round(v, 4) for k, v in test_scores.items()})  # Report the final gold-test metrics, rounded for readability
print(classification_report([tags for _, tags in test], test_preds, digits=4))  # Print the full per-entity-type breakdown (precision/recall/F1/support)

best epoch 4 | val (silver) F1 0.7546
TEST (gold): {'F1_overall': 0.8038, 'F1_PER': 0.8692, 'F1_ORG': 0.713, 'F1_LOC': 0.7933}
              precision    recall  f1-score   support

         LOC     0.8123    0.7752    0.7933       614
         ORG     0.7448    0.6838    0.7130       525
         PER     0.8638    0.8747    0.8692       790

   micro avg     0.8169    0.7911    0.8038      1929
   macro avg     0.8069    0.7779    0.7918      1929
weighted avg     0.8150    0.7911    0.8025      1929



## Log the run

This appends **one** row to `results/results.csv`, so run it once per real run. On Colab/Kaggle the file lives inside the cloned repo. Copy the printed row into your local `results/results.csv` and commit it, along with the notebook.

In [12]:
notes = (f"seed={SEED}; best_epoch={best_epoch}; val_F1={val_scores['F1_overall']:.4f}; "  # Start of the multi-line notes string: run identity and outcome
         f"emb={'fastText cc.hi.300' if USE_FASTTEXT else 'random'}+charBiLSTM; hidden={HIDDEN}; "  # Embedding source and model size
         f"lr={LR}; bs={BATCH_SIZE}; dropout={DROPOUT}; word_dropout={WORD_DROPOUT}"  # Optimization/regularization hyperparameters
         + ("; DEBUG run" if DEBUG else ""))  # Build a human-readable summary of this run's key hyperparameters and outcome, stored alongside the logged metrics

log_result(  # Append one row describing this run to the shared results CSV
    model="BiLSTM-CRF",  # Model family name for this run
    train_lang=LANG,  # Language the model was trained on
    test_lang=LANG,  # Language the model was evaluated on (same as train here, no transfer)
    subset_size=len(train),  # Actual number of training sentences used
    epochs=len(history),  # Number of epochs actually completed (may be less than EPOCHS if early-stopped)
    scores=test_scores,  # The final gold-test metrics dict
    checkpoint_path=BEST,  # Where the best model weights are saved, for later reuse
    notes=notes,  # The hyperparameter summary string built above
)  # End of the log_result(...) call

with open(os.path.join(REPO_ROOT, "results", "results.csv"), encoding="utf-8") as f:  # Reopen the results CSV to show the row that was just appended
    print(f.read().splitlines()[-1])  # Print only the last line (the newly logged run) for a quick visual confirmation

BiLSTM-CRF,hi,hi,100000,7,0.8691823899371068,0.7130089374379344,0.7933333333333333,0.8037924677376876,/content/drive/MyDrive/NaamRekha/phase2/bilstm-crf_hi_n100000_s42/best.pt,seed=42; best_epoch=4; val_F1=0.7546; emb=fastText cc.hi.300+charBiLSTM; hidden=128; lr=0.001; bs=32; dropout=0.5; word_dropout=0.5
